<a href="https://colab.research.google.com/github/LorraineWong/rxplain/blob/main/notebook/rxplain_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rxplain — Pipeline Notebook
*Production entry point · Powered by Gemma 4 · Gemma 4 Good Hackathon 2026*

Run cells in order. Cells 2–5 must be re-run every session after reconnecting.

## 1. Environment Check
Run once to verify GPU availability. Expected: CUDA available, GPU: NVIDIA L4.

In [1]:
# 1. Environment check
# Run once to verify GPU. Expected: CUDA True, GPU Tesla T4.
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

CUDA: True
GPU: NVIDIA L4
PyTorch: 2.10.0+cu128


## 2. Install Dependencies
Run once per session. Installs latest Transformers from source (required for Gemma 4 support).

In [2]:
# 2. Install dependencies — run once per session
import sys, subprocess

# Force reinstall latest transformers from source (required for Gemma 4)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "transformers"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/huggingface/transformers.git",
    "accelerate>=0.29.0",
    "pydantic>=2.6.0",
    "fastapi>=0.110.0",
    "uvicorn==0.29.0",
    "python-multipart>=0.0.9",
    "requests>=2.31.0"], check=True)

print("All dependencies installed.")

print("✅ All dependencies installed.")

All dependencies installed.
✅ All dependencies installed.


In [3]:
import transformers
print(transformers.__version__)

# Check what Gemma4 classes are available
import inspect
gemma4_classes = [name for name in dir(transformers) if 'gemma4' in name.lower() or 'Gemma4' in name]
print(gemma4_classes)

5.8.0.dev0
['Gemma4AssistantConfig', 'Gemma4AssistantForCausalLM', 'Gemma4AssistantPreTrainedModel', 'Gemma4AudioConfig', 'Gemma4AudioFeatureExtractor', 'Gemma4AudioModel', 'Gemma4Config', 'Gemma4ForCausalLM', 'Gemma4ForConditionalGeneration', 'Gemma4ImageProcessor', 'Gemma4ImageProcessorPil', 'Gemma4Model', 'Gemma4PreTrainedModel', 'Gemma4Processor', 'Gemma4TextConfig', 'Gemma4TextModel', 'Gemma4VideoProcessor', 'Gemma4VisionConfig', 'Gemma4VisionModel', 'image_processing_gemma4_fast', 'image_processing_pil_gemma4_fast', 'models.gemma4', 'models.gemma4.configuration_gemma4', 'models.gemma4.feature_extraction_gemma4', 'models.gemma4.image_processing_gemma4', 'models.gemma4.image_processing_pil_gemma4', 'models.gemma4.modeling_gemma4', 'models.gemma4.processing_gemma4', 'models.gemma4.video_processing_gemma4', 'models.gemma4_assistant', 'models.gemma4_assistant.configuration_gemma4_assistant', 'models.gemma4_assistant.modeling_gemma4_assistant']


## 3. Load Model
Run every session (~2 min first time, cached to Google Drive after).

Loads Gemma 4 E4B from cache. First run downloads ~9 GB to your Drive.

In [4]:
# 3. Startup – run every session (~2 min first time, cached after)
import torch, gc
from google.colab import drive, userdata
from huggingface_hub import login
from transformers import AutoProcessor, Gemma4ForConditionalGeneration

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Login to Hugging Face
login(token=userdata.get('HF_TOKEN'))

MODEL_ID = "google/gemma-4-E4B-it"
MODEL_CACHE_DIR = '/content/drive/MyDrive/Kaggle/rxplain-pipeline/models-gemma4'

gc.collect()
torch.cuda.empty_cache()

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=MODEL_CACHE_DIR)
tokenizer = processor

print("Loading model... (~2 min)")
model = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    cache_dir=MODEL_CACHE_DIR
)

print(f"Ready, GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB used")
print(f"Running on: {next(model.parameters()).device}")

Mounted at /content/drive
Loading processor...
Loading model... (~2 min)


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Ready, GPU: 14.8 GB used
Running on: cuda:0


## 4. Load Source Modules
Run every session. Pulls latest code from GitHub and imports all Rxplain modules.

In [5]:
# 4. Load source modules — run every session
import subprocess, sys

# Pull latest from GitHub
subprocess.run(["rm", "-rf", "/content/rxplain"], capture_output=True)
result = subprocess.run(
    ["git", "clone", "--depth=1",
     "https://github.com/LorraineWong/rxplain.git"],
    capture_output=True, text=True
)
print(result.stderr or "✅ Cloned successfully")

# Add src/ to path
for mod in ['schema','dailymed','extract','personalise','server','vision']:
    if mod in sys.modules:
        del sys.modules[mod]

if '/content/rxplain/src' in sys.path:
    sys.path.remove('/content/rxplain/src')
sys.path.insert(0, '/content/rxplain/src')

# Import all modules
from schema import DrugInfo, UserProfile, Severity, FoodAction
from dailymed import get_drug_leaflet
from extract import extract_drug_info_robust
from personalise import personalise, generate_personal_summary
from server import launch
print("✅ All modules loaded!")

Cloning into 'rxplain'...

✅ All modules loaded!


## 5. Launch
Run every session. Starts the FastAPI server and generates a public link.

- Scan a medicine box photo → Gemma 4 identifies the drug name
- Generate a personalised patient guide → Gemma 4 extracts structured data from NIH DailyMed

In [6]:
# 5. Launch — run every session
import torch, gc
import sys
import subprocess
import asyncio
import logging

# Kill any previously running server on port 7860
subprocess.run(["fuser", "-k", "7860/tcp"], capture_output=True)

# Suppress asyncio error logs
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

sys.path.insert(0, '/content/rxplain/src')

import server
import uvicorn
import nest_asyncio

# Inject the already-loaded model from Cell 3 into server
print("[runner] Injecting model into server...")
server._model = model
server._tokenizer = processor
server._processor = processor

gc.collect()
torch.cuda.empty_cache()

nest_asyncio.apply()

# Use Colab built-in proxy
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(7860)")
print(f"\n✅ Legimed is live at: {url}\n")

print("[runner] Starting uvicorn...")
config = uvicorn.Config(server.app, host="0.0.0.0", port=7860, log_level="warning")
uvi_server = uvicorn.Server(config)

try:
    asyncio.get_event_loop().run_until_complete(uvi_server.serve())
except (KeyboardInterrupt, SystemExit):
    pass

[runner] Injecting model into server...

✅ Legimed is live at: https://7860-gpu-l4-s-kkb-ass1a0-3fczuvwyywave-a.asia-southeast1-0.prod.colab.dev

[runner] Starting uvicorn...
[vision] Using Gemma 4 local vision to identify drug...
[vision] Gemma 4 identified: Metformin
Searching DailyMed for: Metformin...
Found DailyMed label: METFORMIN HYDROCHLORIDE TABLET, EXTENDED RELEASE [LUPIN PHARMACEUTICALS, INC.] [2a84c550-aed8-4058-961f-351a5e4cf7a1]
Retrieved 12000 characters, using 6000 relevant characters.


## 6. Evaluate (Optional)
Run after Launch to benchmark the pipeline across 10 common drugs.
Generates a success-rate table for the submission writeup.

Takes ~10–15 minutes. Each drug runs a full Gemma 4 inference cycle.

In [7]:
# 6. Evaluate — optional, run after Cell 5 to generate submission stats
import sys
sys.path.insert(0, '/content/rxplain')
from scripts.evaluate_drugs import evaluate_drugs, print_markdown_table

# Run evaluation on 10 common drugs
results = evaluate_drugs(model, processor)

# Print results as markdown table for README / writeup
print("\n--- Markdown Table ---")
print_markdown_table(results)


[1/10] Testing: warfarin ...
Searching DailyMed for: warfarin...
Found DailyMed label: WARFARIN SODIUM TABLET [NORTHWIND HEALTH COMPANY, LLC] [f86fe457-a1b6-e54d-e053-6294a90a9bc7]
Retrieved 12000 characters, using 4307 relevant characters.
  ✅ 59.38s

[2/10] Testing: metformin ...
Searching DailyMed for: metformin...
Found DailyMed label: METFORMIN HYDROCHLORIDE TABLET, EXTENDED RELEASE [LUPIN PHARMACEUTICALS, INC.] [2a84c550-aed8-4058-961f-351a5e4cf7a1]
Retrieved 12000 characters, using 6000 relevant characters.
  ✅ 72.84s

[3/10] Testing: ibuprofen ...
Searching DailyMed for: ibuprofen...
Found DailyMed label: IBUPROFEN TABLET [NUCARE PHARMACEUTICALS,INC.] [05ce8e8e-0b25-061e-e063-6394a90a2e92]
Retrieved 12000 characters, using 6000 relevant characters.
  ✅ 67.1s

[4/10] Testing: amoxicillin ...
Searching DailyMed for: amoxicillin...
Found DailyMed label: AMOXICILLIN AND CLAVULANATE POTASSIUM TABLET, FILM COATED [COUPLER LLC] [513b6211-c73b-7e66-e063-6394a90a2aa1]
Retrieved 12000 c